# Activation patching trên GPT-2 small: bài toán indirect object identification

"When Mary and John went to the store, John gave a drink to" → GPT-2 small điền " Mary". Wang và cộng sự (2022) đã tìm ra mạch
khoảng 26 head làm việc này. Bạn không đọc kết quả của họ trước: bạn tự dựng cặp prompt **clean** / **corrupted** chỉ khác nhau
một token, đo bằng **logit difference**, rồi patch activation từ run clean vào run corrupted theo từng tầng, từng vị trí, từng
head để tìm ra thông tin đi đường nào. Cuối bài so bản đồ của bạn với paper.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell `explore` tải model và chạy thí nghiệm; test bỏ qua chúng. Cache dùng `run_with_cache` **của bạn** từ bài Basic 01.
- CPU 4 luồng chạy cả notebook trong vài phút; GPU (kernel `ai-gpu`) nhanh hơn nhiều, không bắt buộc.
- Chấm: `uv run pytest tracks/ai-engineer/modules/08-interpretability/exercises/intermediate/02-activation-patching-ioi -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
print(ROOT)

In [ ]:
import random

import torch

## 1. Prompt clean và corrupted (cho sẵn)

Mỗi prompt có dạng `When A and B went to the PLACE, S VERB a OBJECT to`. Trong prompt **clean**, chủ ngữ mệnh đề sau `S` là một
trong hai tên, đáp án là tên còn lại (`IO`, indirect object). Prompt **corrupted** giống hệt, chỉ thay `S` bằng `IO`: đáp án đúng
đảo sang tên kia. Hai prompt khác nhau đúng **một token** (vị trí của `S`, gọi là S2), nên mọi khác biệt trong activation đều bắt
nguồn từ vị trí đó. Mọi tên và từ đều là một token của GPT-2 (có dấu cách đầu), nên mọi prompt cùng độ dài và cùng vị trí.

In [ ]:
NAMES = ["Mary", "John", "Tom", "James", "Dan", "Sid", "Martin", "Amy", "Kate", "Paul", "Mark", "Anna", "Lisa", "Sarah",
         "David", "Michael", "Robert", "Laura", "Emma", "Jack", "Alice", "Bob", "Rachel", "Peter", "Susan", "Kevin", "Linda",
         "Chris", "Daniel", "Jane"]
PLACES = ["store", "park", "school", "office", "restaurant", "hospital", "station", "beach"]
OBJECTS = ["drink", "book", "ball", "gift", "letter", "bag", "key", "phone"]
VERBS = ["gave", "handed", "passed", "sent"]
TEMPLATE = "When {} and {} went to the {}, {} {} a {} to"


def ioi_prompts(n, seed):
    """Trả (clean, corrupt, io, s): hai list prompt và hai list tên (không dấu cách). Nửa số prompt IO đứng trước S trong
    mệnh đề đầu, nửa còn lại IO đứng sau."""
    rng = random.Random(seed)
    clean, corrupt, io, s = [], [], [], []
    for i in range(n):
        a, b = rng.sample(NAMES, 2)
        place, obj, verb = rng.choice(PLACES), rng.choice(OBJECTS), rng.choice(VERBS)
        subj, ind = (b, a) if i % 2 == 0 else (a, b)
        clean.append(TEMPLATE.format(a, b, place, subj, verb, obj))
        corrupt.append(TEMPLATE.format(a, b, place, ind, verb, obj))
        io.append(ind)
        s.append(subj)
    return clean, corrupt, io, s

## 2. Thước đo

- `logit_diff`: $$\text{logit}(IO) - \text{logit}(S)$$ ở vị trí cuối, cho từng prompt. Dùng logit difference thay cho xác suất vì
  nó tuyến tính theo residual stream cuối (direct logit attribution cộng được) và không bị softmax nén khi model đã chắc chắn.
- `normalized_effect`: đưa mọi kết quả patching về cùng thang, 0 là "như corrupted", 1 là "khôi phục hết clean".

In [ ]:
def logit_diff(logits, io, s):
    """logits (B, T, V); io, s (B,) token id. Trả tensor (B,): logit của `io` trừ logit của `s` ở vị trí **cuối**."""
    # TODO
    raise NotImplementedError

In [ ]:
def normalized_effect(patched, clean, corrupt):
    """Hiệu ứng chuẩn hoá của một lần patching (denoising): 0 khi bằng run corrupted, 1 khi khôi phục hết run clean."""
    # TODO
    raise NotImplementedError

## 3. Ba kiểu patching

Cả ba chạy model trên `idx` (run sẽ bị sửa) và thay một activation bằng giá trị lấy từ `source_cache` (cache của run kia, theo
đúng tên khoá của bài Basic 01). **Denoising**: `idx` là corrupted, `source_cache` là clean: "khôi phục riêng activation này có
đủ đưa đáp án về không?". **Noising**: ngược lại: "phá riêng activation này có làm hỏng đáp án không?".

In [ ]:
def patch_resid_pre(model, idx, source_cache, layer, pos=None):
    """Chạy `model(idx)` nhưng thay residual stream đi vào Block `layer` (khoá `resid_pre.layer` của cache bài Basic 01)
    bằng giá trị trong `source_cache`, ở vị trí `pos` (int) hoặc mọi vị trí (`None`). Trả logits (B, T, V), không gradient.
    Gỡ hook trước khi trả, kể cả khi lỗi.
    """
    # TODO
    raise NotImplementedError

In [ ]:
def patch_head(model, idx, source_cache, layer, head, pos=None):
    """Như `patch_resid_pre` nhưng chỉ thay output của **một head** (khoá `z.layer`, shape (B, T, n_head, head_dim)) trước
    `attn.proj` của Block `layer`, ở vị trí `pos` hoặc mọi vị trí. Trả logits.
    """
    # TODO
    raise NotImplementedError

In [ ]:
def patch_sublayer(model, idx, source_cache, kind, layer):
    """Thay **output** của `blocks[layer].attn` (`kind="attn"`, khoá `attn_out.layer`) hoặc `blocks[layer].mlp`
    (`kind="mlp"`, khoá `mlp_out.layer`) ở mọi vị trí bằng giá trị trong `source_cache`. Trả logits.
    """
    # TODO
    raise NotImplementedError

In [ ]:
def resid_patching_sweep(model, idx, source_cache, metric, clean_value, corrupt_value):
    """Bảng (n_layer, T) hiệu ứng chuẩn hoá khi patch `resid_pre.layer` ở từng vị trí: phần tử [l, p] là
    `normalized_effect(metric(patch_resid_pre(model, idx, source_cache, l, p)), clean_value, corrupt_value)`.
    `metric` nhận logits, trả một số float.
    """
    # TODO
    raise NotImplementedError

## 4. Thí nghiệm

Các cell dưới không được chấm. Ghi lại các bảng để làm trắc nghiệm.

In [ ]:
# explore: nạp lớp GPT (module 05), run_with_cache (Basic 01) và direct_logit_attribution (Basic 02); tải GPT-2 small
import importlib.util
import time


def load_file(name, candidates, check):
    for path in candidates:
        spec = importlib.util.spec_from_file_location(name, path)
        module = importlib.util.module_from_spec(spec)
        sys.modules[name] = module
        spec.loader.exec_module(module)
        try:
            check(module)
            print("dùng", path.relative_to(ROOT))
            return module
        except NotImplementedError:
            continue


M05 = ROOT / "tracks/ai-engineer/modules/05-transformers-llm-architecture"
M08 = ROOT / "tracks/ai-engineer/modules/08-interpretability"
gpt_mini = load_file("gpt_mini", [M05 / "exercises/basic/01-gpt-mini/starter/gpt.py", M05 / "solutions/basic/01-gpt-mini/gpt.py"],
                     lambda m: m.GPT(m.GPTConfig(vocab_size=10, block_size=4, n_layer=1, n_head=1, n_embd=8))(torch.zeros(1, 2, dtype=torch.long)))
tiny = gpt_mini.GPT(gpt_mini.GPTConfig(vocab_size=10, block_size=4, n_layer=1, n_head=1, n_embd=8))
hooks = load_file("m08_hooks", [M08 / "exercises/basic/01-hooks-residual-stream/starter/hooks.py", M08 / "solutions/basic/01-hooks-residual-stream/hooks.py"],
                  lambda m: m.run_with_cache(tiny, torch.zeros(1, 3, dtype=torch.long)))
lens = load_file("m08_lens", [M08 / "exercises/basic/02-attention-logit-lens/starter/lens.py", M08 / "solutions/basic/02-attention-logit-lens/lens.py"],
                 lambda m: m.direct_logit_attribution(tiny, {"a": torch.ones(1, 1, 8)}, torch.ones(1, 1, 8), torch.ones(8)))

import json

from transformers import AutoTokenizer, GPT2LMHeadModel


def notebook_namespace(path):
    """Chạy các cell code không gắn tag explore của một notebook, trả namespace (giống cách test nạp notebook)."""
    ns = {}
    for cell in json.loads(Path(path).read_text(encoding="utf-8"))["cells"]:
        if cell["cell_type"] == "code" and "explore" not in cell["metadata"].get("tags", []):
            exec("".join(cell["source"]), ns)
    return ns


def load_gpt2(gpt_mini):
    """GPT-2 small trong lớp GPT bằng gpt2_into_gpt_mini của bạn (bài Intermediate 01); lệch logits so với Hugging Face quá 1e-3
    hoặc lỗi bất kỳ thì dùng lời giải và in cảnh báo."""
    I01 = ROOT / "tracks/ai-engineer/modules/08-interpretability/{}/intermediate/01-gpt2-heads-lens-probe/{}inside_gpt2.ipynb"
    hf = GPT2LMHeadModel.from_pretrained("openai-community/gpt2").eval()
    probe_ids = torch.tensor([[50256, 464, 412, 733, 417, 8765, 318, 5140, 287, 262, 1748, 286]])
    with torch.no_grad():
        ref = hf(probe_ids).logits
    for d, sub in [("exercises", "starter/"), ("solutions", "")]:
        try:
            mdl = notebook_namespace(str(I01).format(d, sub))["gpt2_into_gpt_mini"](gpt_mini, hf)
            with torch.no_grad():
                diff = (mdl(probe_ids)[0] - ref).abs().max().item()
            if diff <= 1e-3:
                print(f"gpt2_into_gpt_mini từ {d} (bài Intermediate 01), lệch logits {diff:.1e}")
                return mdl.eval()
            print(f"cảnh báo: gpt2_into_gpt_mini ở {d} lệch logits {diff:.1e} > 1e-3, chuyển sang lời giải")
        except NotImplementedError:
            continue
        except Exception as e:
            print(f"cảnh báo: gpt2_into_gpt_mini ở {d} lỗi ({type(e).__name__}: {e}), chuyển sang lời giải")
    raise RuntimeError("không nạp được GPT-2")


torch.set_num_threads(4)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained("openai-community/gpt2")
model = load_gpt2(gpt_mini).to(DEVICE).eval()
print("GPT-2 small trên", DEVICE)

In [ ]:
# explore: 32 cặp prompt, token hoá (thêm <|endoftext|> ở đầu), logit difference của hai run
clean_txt, corrupt_txt, io_names, s_names = ioi_prompts(32, seed=0)
BOS = tok.bos_token_id
clean_idx = torch.tensor([[BOS] + tok.encode(p) for p in clean_txt], device=DEVICE)
corrupt_idx = torch.tensor([[BOS] + tok.encode(p) for p in corrupt_txt], device=DEVICE)
io = torch.tensor([tok.encode(" " + n)[0] for n in io_names], device=DEVICE)
s = torch.tensor([tok.encode(" " + n)[0] for n in s_names], device=DEVICE)
POS = tok.convert_ids_to_tokens(clean_idx[0].tolist())
print(clean_txt[0], "|", corrupt_txt[0], "| shape", tuple(clean_idx.shape))
print(" ".join(f"{i}:{t}" for i, t in enumerate(POS)))

clean_logits, clean_cache = hooks.run_with_cache(model, clean_idx)
corrupt_logits, corrupt_cache = hooks.run_with_cache(model, corrupt_idx)
LD_CLEAN = logit_diff(clean_logits, io, s).mean().item()
LD_CORRUPT = logit_diff(corrupt_logits, io, s).mean().item()
p = torch.softmax(clean_logits[:, -1], -1)
rows = torch.arange(len(io))
print(f"logit diff clean {LD_CLEAN:.2f} | corrupted {LD_CORRUPT:.2f} | clean đúng ở {(logit_diff(clean_logits, io, s) > 0).float().mean():.0%} prompt")
print(f"xác suất trung bình (clean): IO {p[rows, io].mean():.3f}, S {p[rows, s].mean():.3f}")
metric = lambda logits: logit_diff(logits, io, s).mean().item()

In [ ]:
# explore: thí nghiệm 1, patch residual stream theo (tầng, vị trí)
start = time.perf_counter()
resid = resid_patching_sweep(model, corrupt_idx, clean_cache, metric, LD_CLEAN, LD_CORRUPT)
print(f"{time.perf_counter() - start:.0f} s")
torch.set_printoptions(precision=2, sci_mode=False, linewidth=160)
print("cột:", " ".join(f"{i}:{t}" for i, t in enumerate(POS)))
print(resid)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(resid.numpy(), cmap="RdBu", vmin=-1, vmax=1)
ax.set_xticks(range(len(POS)), POS, rotation=60); ax.set_ylabel("tầng (resid_pre)")
fig.colorbar(im); ax.set_title("Patch residual stream (clean → corrupted)")
plt.tight_layout(); plt.show()

In [ ]:
# explore: thí nghiệm 2, patch output attention và MLP từng tầng (mọi vị trí), denoising và noising
L, H = model.config.n_layer, model.config.n_head
print("tầng   attn(denoise)  mlp(denoise)   attn(noise)  mlp(noise)")
for l in range(L):
    row = []
    for kind in ["attn", "mlp"]:
        row.append(normalized_effect(metric(patch_sublayer(model, corrupt_idx, clean_cache, kind, l)), LD_CLEAN, LD_CORRUPT))
    for kind in ["attn", "mlp"]:
        row.append(normalized_effect(metric(patch_sublayer(model, clean_idx, corrupt_cache, kind, l)), LD_CLEAN, LD_CORRUPT))
    print(f"{l:4d}   " + "   ".join(f"{v:10.2f}" for v in row))

In [ ]:
# explore: thí nghiệm 3, patch từng head (mọi vị trí), bảng 12 x 12
start = time.perf_counter()
heads = torch.zeros(L, H)
for l in range(L):
    for h in range(H):
        heads[l, h] = normalized_effect(metric(patch_head(model, corrupt_idx, clean_cache, l, h)), LD_CLEAN, LD_CORRUPT)
print(f"{time.perf_counter() - start:.0f} s")
print(heads)
for name, largest in [("dương", True), ("âm", False)]:
    v, i = heads.flatten().topk(8, largest=largest)
    print(f"head {name} mạnh nhất:", "  ".join(f"{j // H}.{j % H}={x:+.2f}" for x, j in zip(v.tolist(), i.tolist())))
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(heads.numpy(), cmap="RdBu", vmin=-0.5, vmax=0.5)
ax.set_xlabel("head"); ax.set_ylabel("tầng"); fig.colorbar(im); ax.set_title("Patch từng head")
plt.tight_layout(); plt.show()

In [ ]:
# explore: thí nghiệm 4, direct logit attribution (hàm của bài Basic 02) của từng head ở vị trí cuối (run clean) và head đó nhìn vào đâu
direction = (model.lm_head.weight[io] - model.lm_head.weight[s]).detach()
final = clean_cache[f"resid_post.{L - 1}"]
contrib = {}
for l in range(L):
    hc = hooks.head_contributions(model, clean_cache, l)
    for h in range(H):
        contrib[f"{l}.{h}"] = hc[:, :, h]
dla = {k: 0.0 for k in contrib}
for b in range(len(io)):
    one = lens.direct_logit_attribution(model, {k: v[b:b + 1] for k, v in contrib.items()}, final[b:b + 1], direction[b])
    for k, v in one.items():
        dla[k] += v[0, -1].item() / len(io)
top = sorted(dla.items(), key=lambda kv: -abs(kv[1]))[:8]
print("DLA (logit diff, vị trí cuối):", "  ".join(f"{k}={v:+.2f}" for k, v in top))
pos_names = {"IO/S lần 1": [2, 4], "S2": [10], "cuối": [14]}
for k, _ in top[:5]:
    l, h = map(int, k.split("."))
    pat = clean_cache[f"pattern.{l}"][:, h, -1]
    io_pos = torch.where(clean_idx[:, 2] == io, 2, 4)
    print(f"head {k}: attention từ vị trí cuối tới IO {pat[rows, io_pos].mean():.2f}, tới S1 {pat[rows, 6 - io_pos].mean():.2f}, tới S2 {pat[:, 10].mean():.2f}")

## 5. Đọc kết quả

Trả lời cho chính mình (rồi làm phần **trắc nghiệm** trong app):

1. Bảng residual: ở các tầng đầu, chỉ cột nào sáng? Từ tầng nào thông tin "chuyển" sang vị trí cuối? Điều đó nói gì về việc phần
   nào của mạch chạy ở vị trí S2, phần nào ở vị trí cuối?
2. Bảng attention/MLP: tầng nào có hiệu ứng lớn nhất? Vì sao MLP tầng 0 lại có hiệu ứng lớn dù nó chỉ nhìn một vị trí? Denoising
   và noising có cho cùng thứ hạng không?
3. Bảng head: head dương mạnh nhất ở những tầng nào, có head âm không? So với DLA: head có hiệu ứng patching lớn có phải lúc nào
   cũng có DLA lớn không, và ngược lại? (Gợi ý: một head có thể tác động qua head khác chứ không trực tiếp lên logit.)
4. So với Wang và cộng sự (2022): name mover, negative name mover, S-inhibition, induction, duplicate token. Bản đồ của bạn khớp
   ở đâu, khác ở đâu, và vì sao 32 prompt một template chưa đủ để gọi đó là "mạch IOI"?